# 1. Introducción

**Problema industrial:** Clasificación automática de estados normal/falla en equipos rotativos.

**Activo analizado:** PUMP101 — variables: temperatura, vibración, presión, corriente.

**Origen de datos:** Dataset multivariable etiquetado desde PI y eventos de falla (simulado).

**Objetivo del análisis:** Entrenar un clasificador Random Forest para anticipar fallas.


# 2. Carga de librerías

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

LAB_DIR = Path.cwd()
os.chdir(LAB_DIR)
OUTPUT_DIR = LAB_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
EXCEL_DIR = LAB_DIR / "excel"
DATA_PATH = LAB_DIR / "data" / "datos_exportados_PI.csv"
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.model_selection import train_test_split


# 3. Lectura de datos PI System

Simulamos una exportación del historiador PI con columnas: `Timestamp`, `Tag`, `Value`, `Unit`, `Quality`.

In [ ]:
df_pi = pd.read_csv(DATA_PATH, parse_dates=["Timestamp"])
print(f"Registros cargados: {len(df_pi):,}")
df_pi.head(10)


# 4. Exploración del dato

In [ ]:
print("Columnas:", df_pi.columns.tolist())
print("\nEstadísticas por tag:")
display(df_pi.groupby("Tag")["Value"].describe())

calidad = df_pi["Quality"].value_counts(normalize=True) * 100
print("\nCalidad del dato (%):")
print(calidad.round(2))

faltantes = df_pi["Value"].isna().sum()
print(f"\nValores faltantes: {faltantes}")

df_good = df_pi[df_pi["Quality"] == "GOOD"].copy()
tendencia = df_good.groupby("Tag")["Value"].agg(["mean", "std", "min", "max"])
print("\nTendencia central por tag:")
display(tendencia)


# 5. Análisis matemático

Feature engineering y clasificación binaria falla/normal.

In [ ]:
wide = df_good.pivot_table(index="Timestamp", columns="Tag", values="Value", aggfunc="mean").dropna()
features = wide[["PUMP101.BEARING_TEMP", "PUMP101.VIBRATION_RMS", "PUMP101.DISCHARGE_PRESS", "PUMP101.MOTOR_CURRENT"]]

# Etiqueta: falla si vibración > percentil 90
umbral_falla = features["PUMP101.VIBRATION_RMS"].quantile(0.90)
y = (features["PUMP101.VIBRATION_RMS"] > umbral_falla).astype(int)

X_train, X_test, y_train, y_test = train_test_split(features, y, test_size=0.25, random_state=42, stratify=y)
modelo = RandomForestClassifier(n_estimators=100, random_state=42)
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)
y_prob = modelo.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["Normal", "Falla"]))
importancia = pd.DataFrame({"Feature": features.columns, "Importancia": modelo.feature_importances_}).sort_values("Importancia", ascending=False)
resultados_export = importancia
display(resultados_export)


# 6. Visualizaciones

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

cm = confusion_matrix(y_test, y_pred)
axes[0].imshow(cm, cmap="Blues")
axes[0].set_xticks([0, 1]); axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(["Normal", "Falla"]); axes[0].set_yticklabels(["Normal", "Falla"])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i,j]>cm.max()/2 else "black")
axes[0].set_title("Matriz de confusión")

axes[1].barh(importancia["Feature"], importancia["Importancia"], color="teal")
axes[1].set_title("Importancia de features")
axes[1].invert_yaxis()

fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[2].plot(fpr, tpr, label=f"AUC={auc(fpr, tpr):.3f}")
axes[2].plot([0, 1], [0, 1], "k--")
axes[2].set_xlabel("FPR"); axes[2].set_ylabel("TPR")
axes[2].set_title("Curva ROC")
axes[2].legend()


# 7. Exportación

In [ ]:
resultados_path = OUTPUT_DIR / "resultado_analisis.csv"
graficos_path = OUTPUT_DIR / "graficos.png"
excel_resultado = EXCEL_DIR / "modelo_resultado.xlsx"

resultados_export.to_csv(resultados_path, index=False)
with pd.ExcelWriter(excel_resultado, engine="openpyxl") as writer:
    resultados_export.to_excel(writer, sheet_name="Importancia", index=False)
    pd.DataFrame(classification_report(y_test, y_pred, output_dict=True)).to_excel(writer, sheet_name="Metricas")

plt.tight_layout()
plt.savefig(graficos_path, dpi=150, bbox_inches="tight")
print(f"CSV exportado: {resultados_path}")
print(f"Gráficos exportados: {graficos_path}")
print(f"Excel exportado: {excel_resultado}")


# 8. Interpretación ingenieril

## Interpretación para mantenimiento

La vibración RMS es el predictor dominante de falla inminente. Integrar este modelo como alerta secundaria en PI AF permite priorizar órdenes de trabajo antes del evento crítico.
